# White's first-move advantage on Lichess

This notebook runs the analysis in this repository on Google Colab. It downloads monthly game
files from the [Lichess Open Database](https://database.lichess.org), measures White's advantage
in rating points, and saves the results.

The monthly files are tens of GB each, so they are never saved to disk: each file is streamed
from Lichess straight into the program, which reads it as it downloads. Memory use stays small.

**What you need:** a Google account and nothing else. Everything is installed in the Colab session.

**What you get:** a `results.zip` with
- `counts.csv`: every game counted by opening, speed, rating bracket and rating gap. Every other
  result can be rebuilt from this file in seconds (step 5), without downloading the games again.
- `output.csv` and `openings.csv`: White's advantage by speed and rating bracket, and by opening.
- `summary.txt`: everything the program printed.
- `dashboard.html`: an interactive dashboard; open it in a browser.

Run the cells in order and keep the tab open while it runs. The full run takes a while, because
it's limited by the download speed from Lichess.

## 1. Install zstd and compile the program

Installs the zstd compression library, clones this repository into `repo/` and compiles
`scripts/white_advantage.cpp`. Running this cell again starts from a fresh copy of the repository
(and deletes anything made in an earlier run of this session).

In [ ]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y libzstd-dev > /dev/null
!rm -rf repo && git clone -q https://github.com/Sahil-M-Deo/The-Perks-of-Being-a-First-Mover.git repo
!g++ -O2 -std=c++17 -pthread repo/scripts/white_advantage.cpp -o repo/scripts/white_advantage -lzstd && echo "compiled"

## 2. Choose the months

`MONTHS` lists the months to analyse, as `YYYY-MM`. The available months are listed at
[database.lichess.org](https://database.lichess.org). The results in this repository use
January and February 2024.

`MAX_GAMES = 0` reads every game. A good first step is a test run with `MAX_GAMES = 1000000`:
it takes a few minutes and checks that everything works (see the checks under step 3).

In [ ]:
MONTHS = ["2024-01", "2024-02"]
MAX_GAMES = 0

## 3. Run

All months are downloaded and read at the same time, each through a named pipe (`mkfifo`), so
the compressed files never sit on disk. The results are written to `repo/output/final/`.

**Checks after it finishes:**
- The "games read" for each month should match the Games column on database.lichess.org (for a
  test run, it should equal `MAX_GAMES`). If it's lower, the download was cut off; run the cell again.
- In the "By opening" table every opening should have games. A line saying "no games (check the
  name...)" means Lichess spells that opening differently in the chosen months; the names are
  listed in `scripts/src/openings.h`.
- On a small test run some lines may show `nan`: some rating brackets don't have enough games
  yet. This goes away in a full run.

In [ ]:
files = [f"lichess_db_standard_rated_{m}.pgn.zst" for m in MONTHS]
lines = ["mkdir -p repo/output/final", "cd repo/output/final", "rm -f lichess_db_*.pgn.zst"]
for f in files:
    # a named pipe with the real file name, so the program knows the month and that it is .zst
    lines.append(f"mkfifo {f}")
    # -f: on an HTTP error write nothing, instead of feeding an HTML error page to zstd
    lines.append(f"curl -fsSL https://database.lichess.org/standard/{f} > {f} &")
extra = f"--max-games {MAX_GAMES}" if MAX_GAMES > 0 else ""
lines.append(f"../../scripts/white_advantage {' '.join(files)} {extra} | tee summary.txt")
lines.append("rm -f lichess_db_*.pgn.zst")
open("run.sh", "w").write("\n".join(lines) + "\n")

# run with ! (not subprocess) so the output shows up live in the notebook
!bash run.sh

## 4. Download the results

Zips the five result files into `results.zip` and downloads it.

In [ ]:
!cd repo/output/final && zip -q ../../../results.zip counts.csv output.csv openings.csv summary.txt dashboard.html; ls -la ../../../results.zip
from google.colab import files as colab_files
colab_files.download("results.zip")

## 5. (Optional) Rebuild the results, dashboard and report from `counts.csv`

Rebuilds every table, test and the dashboard from `repo/output/final/counts.csv`, and builds the
report `reports/final/finalReport.pdf`, in seconds and without downloading any games. It uses the
`counts.csv` from step 3 if you ran it in this session, or otherwise the one included in the
repository (run step 1 first). Installing LaTeX for the report takes a few minutes.

In [ ]:
!apt-get -qq install -y texlive-latex-recommended texlive-latex-extra texlive-fonts-recommended texlive-pictures > /dev/null && echo "LaTeX installed"
!cd repo/output/final && ../../scripts/white_advantage --from-counts counts.csv --report ../../reports/final/report_numbers.tex
!cd repo && zip -q -j ../report.zip reports/final/finalReport.pdf output/final/dashboard.html output/final/output.csv output/final/openings.csv
from google.colab import files as colab_files
colab_files.download("report.zip")